In [1]:
import os
import csv
import itertools
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
import matplotlib.pyplot as plt

# ------------------------ ViT Block ------------------------
class ViT(nn.Module):
    def __init__(self, in_channels: int = 512, patch_size: int = 4, emb_size: int = 128, num_classes: int = 39):
        super().__init__()
        self.patch_size = patch_size
        self.conv = nn.Conv2d(in_channels, patch_size ** 2, kernel_size=patch_size, stride=patch_size)
        self.patch_emb = nn.Linear(patch_size ** 2, emb_size)
        self.cls_token = nn.Parameter(torch.randn(1, 1, emb_size))
        self.pos_emb = None
        enc_layer = nn.TransformerEncoderLayer(d_model=emb_size, nhead=4, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=4)
        self.classifier = nn.Linear(emb_size, num_classes)

    def forward(self, x):
        x = self.conv(x)  # (B, P^2, 7, 7)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)  # (B, 49, P^2)
        x = self.patch_emb(x)  # (B, 49, E)
        cls = self.cls_token.expand(B, 1, -1)
        x = torch.cat([cls, x], dim=1)  # (B, 50, E)
        if self.pos_emb is None or self.pos_emb.size(1) != x.size(1):
            self.pos_emb = nn.Parameter(torch.randn(1, x.size(1), x.size(2), device=x.device))
        x = x + self.pos_emb
        x = self.encoder(x)
        return self.classifier(x[:, 0])

# -------------------- ResNet50 + ViT Model --------------------
class ResNet50_ViT(nn.Module):
    def __init__(self, num_classes: int = 39, freeze_backbone: bool = True):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.stem = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
        )
        if freeze_backbone:
            for p in self.stem.parameters():
                p.requires_grad = False
        self.vit = ViT(in_channels=512, patch_size=4, emb_size=128, num_classes=num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.vit(x)
        return x

# --------------------- Configuration ---------------------
DATA_DIR = r"E:\Learning\UNSW\Term2\9444\group_project\data\split_with_713"
OUTPUT_DIR = r"E:\Learning\UNSW\Term2\9444\group_project\outputs\plot\final\resnet50_with_trans2"
NUM_CLASSES = 39
BATCH_SIZE = 64
NUM_WORKERS = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_EPOCHS = 30
WEIGHT_DECAY = 1e-5
LR = 1e-4
PATIENCE = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.backends.cudnn.benchmark = True

# -------------------- Data Transforms --------------------
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ----------------------- Dataloaders -----------------------
def get_dataloaders(data_dir: str, batch_size: int, num_workers: int):
    train_ds = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transform)
    val_ds = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=val_transform)
    test_ds = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=val_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader

# ------------------------- Model -------------------------
model = ResNet50_ViT(num_classes=NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler("cuda")

# -------------------- Helper Function --------------------

def evaluate_predictions(model: nn.Module, loader: DataLoader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())
    return all_labels, all_preds

# ---------------------- Training Loop ----------------------

def train():
    train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, BATCH_SIZE, NUM_WORKERS)

    best_val_acc = 0.0
    epochs_no_improve = 0
    epochs_list, train_losses, val_accs, val_f1s = [], [], [], []

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with autocast("cuda"):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * imgs.size(0)

        scheduler.step()
        train_loss = running_loss / len(train_loader.dataset)

        true_val, pred_val = evaluate_predictions(model, val_loader)
        val_acc = accuracy_score(true_val, pred_val)
        val_f1 = f1_score(true_val, pred_val, average="macro")

        epochs_list.append(epoch)
        train_losses.append(train_loss)
        val_accs.append(val_acc)
        val_f1s.append(val_f1)

        print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_resnet50_vit.pth"))
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch}.")
                break

    # ---------------- Final Test Evaluation ----------------
    y_true, y_pred = evaluate_predictions(model, test_loader)
    test_acc = accuracy_score(y_true, y_pred)
    test_prec = precision_score(y_true, y_pred, average="macro")
    test_rec = recall_score(y_true, y_pred, average="macro")
    test_f1 = f1_score(y_true, y_pred, average="macro")

    print("\n===== FINAL TEST METRICS =====")
    print(f"Accuracy : {test_acc:.4f}")
    print(f"Precision: {test_prec:.4f}")
    print(f"Recall   : {test_rec:.4f}")
    print(f"F1-Score : {test_f1:.4f}\n")

    # ---------------- Confusion Matrix ----------------
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.title("Confusion Matrix — Test Set")
    plt.colorbar(shrink=0.8)
    tick_marks = range(NUM_CLASSES)
    plt.xticks(tick_marks)
    plt.yticks(tick_marks)
    thresh = cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], "d"), ha="center", va="center", color="white" if cm[i, j] > thresh else "black")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix_test.png"), dpi=300)
    plt.close()

    # ---------------- Save Metrics CSV ----------------
    with open(os.path.join(OUTPUT_DIR, "metrics.csv"), "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "val_acc", "val_f1"])
        for e, l, a, f1_value in zip(epochs_list, train_losses, val_accs, val_f1s):
            writer.writerow([e, l, a, f1_value])

    # ---------------- Plot Curves ----------------
    # Training Loss
    plt.figure()
    plt.plot(epochs_list, train_losses, marker="o")
    plt.title("Training Loss vs. Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_loss.png"), dpi=300)
    plt.close()

    # Validation Accuracy
    plt.figure()
    plt.plot(epochs_list, val_accs, marker="o")
    plt.title("Validation Accuracy vs. Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "validation_accuracy.png"), dpi=300)
    plt.close()

    # Validation F1
    plt.figure()
    plt.plot(epochs_list, val_f1s, marker="o")
    plt.title("Validation F1-Score vs. Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("F1-Score")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "validation_f1.png"), dpi=300)
    plt.close()

if __name__ == "__main__":
    train()


Epoch  1/30 | Loss: 1.9221 | Val Acc: 0.7593 | Val F1: 0.6914
Epoch  2/30 | Loss: 0.9359 | Val Acc: 0.8121 | Val F1: 0.7716
Epoch  3/30 | Loss: 0.6603 | Val Acc: 0.8769 | Val F1: 0.8477
Epoch  4/30 | Loss: 0.5232 | Val Acc: 0.9209 | Val F1: 0.9031
Epoch  5/30 | Loss: 0.4399 | Val Acc: 0.9225 | Val F1: 0.9111
Epoch  6/30 | Loss: 0.3989 | Val Acc: 0.9239 | Val F1: 0.9122
Epoch  7/30 | Loss: 0.3596 | Val Acc: 0.9158 | Val F1: 0.9040
Epoch  8/30 | Loss: 0.3310 | Val Acc: 0.9319 | Val F1: 0.9181
Epoch  9/30 | Loss: 0.3009 | Val Acc: 0.9578 | Val F1: 0.9485
Epoch 10/30 | Loss: 0.2898 | Val Acc: 0.9493 | Val F1: 0.9396
Epoch 11/30 | Loss: 0.2756 | Val Acc: 0.9589 | Val F1: 0.9512
Epoch 12/30 | Loss: 0.2457 | Val Acc: 0.9560 | Val F1: 0.9467
Epoch 13/30 | Loss: 0.2387 | Val Acc: 0.9624 | Val F1: 0.9542
Epoch 14/30 | Loss: 0.2212 | Val Acc: 0.9635 | Val F1: 0.9565
Epoch 15/30 | Loss: 0.2074 | Val Acc: 0.9713 | Val F1: 0.9648
Epoch 16/30 | Loss: 0.1994 | Val Acc: 0.9535 | Val F1: 0.9484
Epoch 17